# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kr8457/FlyRank-AI-ML-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: 1 row = 1 unique content_hash_id per calendar month window

Time Window: Mid-panel observation period (e.g., report_date spanning active 30-day performance windows).

Contract Rule: Aggregating daily Search Console (gsc_*) and GA4 (ga4_*) performance logs into monthly page-level snapshots so the model evaluates historical performance per content piece.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import requests
import io
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
url = "https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance_sample.parquet"
headers = {"Authorization": f"Bearer {hf_token}"}

response = requests.get(url, headers=headers)
if response.status_code == 200:
    df = pd.read_parquet(io.BytesIO(response.content))

    # Verify Unit of Analysis & Time Window
    min_date = df['report_date'].min()
    max_date = df['report_date'].max()
    unique_urls = df['content_hash_id'].nunique()

    print(f"Date Range: {min_date} to {max_date}")
    print(f"Unique URLs/Content IDs: {unique_urls:,}")
    print(f"Total Daily Rows Loaded: {len(df):,}")

Date Range: 2026-06-01 to 2026-06-30
Unique URLs/Content IDs: 409,205
Total Daily Rows Loaded: 11,694,072


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

1. Feature Fields: gsc_clicks, gsc_impressions, gsc_avg_position, ga4_sessions, ga4_total_engagement_sec, sessions_organic, sessions_ai (Used to construct 30-day historical features).
2. Label Field: traffic_decay_risk (Binary target: 1 if future month clicks drop $\ge 20\%$, else 0).
3. Context Fields: content_hash_id, report_date (Identifiers used for grouping and rolling time joins).
4. Excluded Fields: Paid/direct traffic columns (sessions_paid, sessions_direct).Why Excluded: Paid campaigns introduce external ad budget spikes that mask organic SEO signals and create noise.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Bucket definitions
feature_cols = ['gsc_clicks', 'gsc_impressions', 'gsc_avg_position', 'ga4_total_engagement_sec', 'sessions_organic']
label_col = 'traffic_decay_risk'
context_cols = ['content_hash_id', 'report_date']
excluded_cols = ['sessions_paid', 'sessions_direct']

print("Field Buckets Defined:")
print(f"Features: {len(feature_cols)} fields")
print(f"Label: {label_col}")
print(f"Excluded: {excluded_cols}")

Field Buckets Defined:
Features: 5 fields
Label: traffic_decay_risk
Excluded: ['sessions_paid', 'sessions_direct']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

We verify data health by checking for duplicate daily records per URL, missing null values in critical search features, and verifying non-zero impression distributions.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check missing values in feature set
missing_summary = df[feature_cols].isnull().sum()
print("Missing Values per Feature Column:")
print(missing_summary)

# Verify zero-click versus non-zero impression ratio
zero_clicks = (df['gsc_clicks'] == 0).sum()
print(f"\nZero-Click Daily Records: {zero_clicks:,} ({(zero_clicks/len(df))*100:.2f}%)")

Missing Values per Feature Column:
gsc_clicks                        0
gsc_impressions                   0
gsc_avg_position            7815170
ga4_total_engagement_sec    2397428
sessions_organic            2397428
dtype: int64

Zero-Click Daily Records: 11,246,705 (96.17%)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data Limitations:

GSC Delay & Sampling: Search Console data represents aggregated search impressions, not raw server request logs.

Short History Window: Newly published URLs without 30+ days of historical data cannot be evaluated for traffic decay.

Non-Attributed AI Traffic: AI referral traffic (sessions_ai) relies on standard referral headers, which may underestimate AI search engines that strip referrer parameters.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Document and print limits programmatically
limits = [
    "Unbalanced history: Cold-start pages with <30 days history excluded",
    "Third-party tracking: Referral stripping in AI answer engines",
    "GSC aggregation: Position averages are non-linear"
]

print("Documented Data Limitations:")
for i, limit in enumerate(limits, 1):
    print(f"{i}. {limit}")

Documented Data Limitations:
1. Unbalanced history: Cold-start pages with <30 days history excluded
2. Third-party tracking: Referral stripping in AI answer engines
3. GSC aggregation: Position averages are non-linear


## Self-check
Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.